# Graph Visualization with Sector Labels

This notebook allows you to visualize stock correlation graphs with sector-colored nodes.

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os

# For interactive file selection
from ipywidgets import interact, Dropdown
import ipywidgets as widgets

In [ ]:
# Configuration - Change this path to your graph file
GRAPH_FILE = "data/graph_structure/pearson/0.5.csv"

# Alternative examples:
# GRAPH_FILE = "data/graph_structure/pearson/0.3.csv"
# GRAPH_FILE = "data/graph_structure/pearson/0.45.csv"
# GRAPH_FILE = "data/graph_structure/identity.csv"
# GRAPH_FILE = "data/graph_structure/cvx_opt/100_0.01_cvx.csv"  # if it exists

SECTOR_FILE = "gephi_bbg_sectors.csv"

In [ ]:
def load_adjacency_matrix(filepath):
    """Load adjacency matrix from CSV file."""
    df = pd.read_csv(filepath, index_col=0)
    return df

def load_sector_labels(filepath):
    """Load sector labels from CSV file."""
    df = pd.read_csv(filepath)
    # Create mapping from ticker to sector
    sector_map = dict(zip(df['Id'], df['Sector']))
    return sector_map

def create_graph_from_adjacency(adj_df):
    """Create NetworkX graph from adjacency matrix DataFrame."""
    G = nx.Graph()
    
    # Add nodes
    tickers = adj_df.index.tolist()
    G.add_nodes_from(tickers)
    
    # Add edges where adjacency value is 1 (excluding self-loops)
    for i, ticker1 in enumerate(tickers):
        for j, ticker2 in enumerate(tickers):
            if i < j and adj_df.iloc[i, j] == 1:
                G.add_edge(ticker1, ticker2)
    
    return G

In [ ]:
# Load data
print(f"Loading graph from: {GRAPH_FILE}")
adj_matrix = load_adjacency_matrix(GRAPH_FILE)
sector_map = load_sector_labels(SECTOR_FILE)

print(f"Number of stocks: {len(adj_matrix)}")
print(f"Adjacency matrix shape: {adj_matrix.shape}")

# Create graph
G = create_graph_from_adjacency(adj_matrix)
print(f"\nGraph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.4f}")

In [ ]:
# Define sector colors
SECTOR_COLORS = {
    'Information Technology': '#1f77b4',  # blue
    'Healthcare': '#2ca02c',              # green
    'Financials': '#d62728',              # red
    'Consumer Discretionary': '#ff7f0e',  # orange
    'Consumer Staples': '#9467bd',        # purple
    'Industrials': '#8c564b',             # brown
    'Communication Services': '#e377c2',  # pink
    'Energy': '#7f7f7f',                  # gray
    'Utilities': '#bcbd22',               # olive
    'Real Estate': '#17becf',             # cyan
    'Materials': '#aec7e8',               # light blue
}

def get_node_colors(G, sector_map):
    """Get color for each node based on sector."""
    colors = []
    for node in G.nodes():
        sector = sector_map.get(node, 'Unknown')
        color = SECTOR_COLORS.get(sector, '#333333')
        colors.append(color)
    return colors

In [ ]:
def visualize_graph(G, sector_map, title="Stock Correlation Graph", 
                    figsize=(16, 12), node_size=500, font_size=8,
                    layout='spring', show_labels=True, edge_alpha=0.3):
    """
    Visualize the graph with sector-colored nodes.
    
    Parameters:
    - layout: 'spring', 'kamada_kawai', 'circular', 'shell', or 'spectral'
    """
    fig, ax = plt.subplots(figsize=figsize)
    
    # Choose layout
    if layout == 'spring':
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    elif layout == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(G)
    elif layout == 'circular':
        pos = nx.circular_layout(G)
    elif layout == 'shell':
        # Group by sector for shell layout
        sectors = {}
        for node in G.nodes():
            sector = sector_map.get(node, 'Unknown')
            if sector not in sectors:
                sectors[sector] = []
            sectors[sector].append(node)
        shells = list(sectors.values())
        pos = nx.shell_layout(G, nlist=shells)
    elif layout == 'spectral':
        pos = nx.spectral_layout(G)
    else:
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    # Get node colors
    node_colors = get_node_colors(G, sector_map)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, alpha=edge_alpha, edge_color='gray', ax=ax)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_size, 
                          alpha=0.9, ax=ax)
    
    # Draw labels
    if show_labels:
        nx.draw_networkx_labels(G, pos, font_size=font_size, font_weight='bold', ax=ax)
    
    # Create legend
    sectors_in_graph = set(sector_map.get(n, 'Unknown') for n in G.nodes())
    legend_elements = [plt.scatter([], [], c=SECTOR_COLORS.get(s, '#333333'), 
                                   s=100, label=s) 
                       for s in sorted(sectors_in_graph)]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=9, 
              title='Sectors', title_fontsize=10)
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    
    return fig, ax

In [ ]:
# Visualize with spring layout
fig, ax = visualize_graph(
    G, sector_map, 
    title=f"Graph: {os.path.basename(GRAPH_FILE)} ({G.number_of_edges()} edges)",
    layout='spring',
    node_size=600,
    font_size=7
)
plt.show()

In [ ]:
# Try different layouts
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

layouts = ['spring', 'kamada_kawai', 'circular', 'shell']

for ax, layout in zip(axes.flat, layouts):
    # Choose layout
    if layout == 'spring':
        pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    elif layout == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(G)
    elif layout == 'circular':
        pos = nx.circular_layout(G)
    elif layout == 'shell':
        sectors = {}
        for node in G.nodes():
            sector = sector_map.get(node, 'Unknown')
            if sector not in sectors:
                sectors[sector] = []
            sectors[sector].append(node)
        shells = list(sectors.values())
        pos = nx.shell_layout(G, nlist=shells)
    
    node_colors = get_node_colors(G, sector_map)
    nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color='gray', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=300, alpha=0.9, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=6, ax=ax)
    ax.set_title(f'{layout.replace("_", " ").title()} Layout', fontsize=12)
    ax.axis('off')

plt.suptitle(f"Graph Layouts: {os.path.basename(GRAPH_FILE)}", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show sector statistics
print("Sector Distribution in Graph:")
print("=" * 40)

sector_counts = {}
for node in G.nodes():
    sector = sector_map.get(node, 'Unknown')
    sector_counts[sector] = sector_counts.get(sector, 0) + 1

for sector, count in sorted(sector_counts.items(), key=lambda x: -x[1]):
    print(f"  {sector}: {count} stocks")

In [ ]:
# Analyze connectivity by sector
print("\nEdge Analysis by Sector:")
print("=" * 40)

# Count intra-sector vs inter-sector edges
intra_sector = 0
inter_sector = 0
sector_pairs = {}

for u, v in G.edges():
    s1 = sector_map.get(u, 'Unknown')
    s2 = sector_map.get(v, 'Unknown')
    
    if s1 == s2:
        intra_sector += 1
    else:
        inter_sector += 1
        pair = tuple(sorted([s1, s2]))
        sector_pairs[pair] = sector_pairs.get(pair, 0) + 1

total_edges = G.number_of_edges()
print(f"Intra-sector edges: {intra_sector} ({100*intra_sector/total_edges:.1f}%)")
print(f"Inter-sector edges: {inter_sector} ({100*inter_sector/total_edges:.1f}%)")

print("\nTop 10 Inter-Sector Connections:")
for pair, count in sorted(sector_pairs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {pair[0]} <-> {pair[1]}: {count} edges")

In [ ]:
# Node degree analysis
degrees = dict(G.degree())
degree_df = pd.DataFrame([
    {'Ticker': node, 'Degree': deg, 'Sector': sector_map.get(node, 'Unknown')}
    for node, deg in degrees.items()
])

print("Top 15 Most Connected Stocks:")
print(degree_df.sort_values('Degree', ascending=False).head(15).to_string(index=False))

print("\nLeast Connected Stocks (excluding isolated):")
connected = degree_df[degree_df['Degree'] > 0]
print(connected.sort_values('Degree').head(10).to_string(index=False))

In [ ]:
# Plot degree distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1 = axes[0]
ax1.hist(degree_df['Degree'], bins=20, edgecolor='black', alpha=0.7)
ax1.set_xlabel('Degree')
ax1.set_ylabel('Count')
ax1.set_title('Degree Distribution')
ax1.axvline(degree_df['Degree'].mean(), color='red', linestyle='--', 
            label=f"Mean: {degree_df['Degree'].mean():.1f}")
ax1.legend()

# By sector
ax2 = axes[1]
sector_avg_degree = degree_df.groupby('Sector')['Degree'].mean().sort_values(ascending=True)
colors = [SECTOR_COLORS.get(s, '#333333') for s in sector_avg_degree.index]
sector_avg_degree.plot(kind='barh', ax=ax2, color=colors)
ax2.set_xlabel('Average Degree')
ax2.set_title('Average Degree by Sector')

plt.tight_layout()
plt.show()

In [ ]:
# Interactive: Compare different graph files
def list_available_graphs():
    """List all available graph CSV files."""
    graphs = []
    
    # Check pearson directory
    pearson_dir = "data/graph_structure/pearson"
    if os.path.exists(pearson_dir):
        for f in os.listdir(pearson_dir):
            if f.endswith('.csv'):
                graphs.append(os.path.join(pearson_dir, f))
    
    # Check cvx_opt directory
    cvx_dir = "data/graph_structure/cvx_opt"
    if os.path.exists(cvx_dir):
        for f in os.listdir(cvx_dir):
            if f.endswith('.csv'):
                graphs.append(os.path.join(cvx_dir, f))
    
    # Check identity
    identity = "data/graph_structure/identity.csv"
    if os.path.exists(identity):
        graphs.append(identity)
    
    return sorted(graphs)

available_graphs = list_available_graphs()
print("Available graph files:")
for g in available_graphs:
    print(f"  - {g}")

In [ ]:
# Function to quickly visualize any graph
def quick_viz(graph_path):
    """Quickly visualize a graph file."""
    adj = load_adjacency_matrix(graph_path)
    G_new = create_graph_from_adjacency(adj)
    
    print(f"Graph: {graph_path}")
    print(f"  Nodes: {G_new.number_of_nodes()}, Edges: {G_new.number_of_edges()}")
    print(f"  Density: {nx.density(G_new):.4f}")
    
    fig, ax = visualize_graph(
        G_new, sector_map,
        title=f"{os.path.basename(graph_path)} ({G_new.number_of_edges()} edges)",
        layout='spring',
        node_size=500,
        font_size=7
    )
    plt.show()
    
    return G_new

# Example: visualize a different graph
# quick_viz("data/graph_structure/pearson/0.3.csv")

In [ ]:
# Compare edge counts across thresholds
if available_graphs:
    comparison = []
    for gpath in available_graphs:
        try:
            adj = load_adjacency_matrix(gpath)
            G_temp = create_graph_from_adjacency(adj)
            comparison.append({
                'Graph': os.path.basename(gpath),
                'Edges': G_temp.number_of_edges(),
                'Density': nx.density(G_temp)
            })
        except Exception as e:
            print(f"Error loading {gpath}: {e}")
    
    comp_df = pd.DataFrame(comparison)
    print("\nGraph Comparison:")
    print(comp_df.to_string(index=False))